# Prueba Spark + MovieLens

Generado: 2026-04-17 15:34:42

Este notebook prueba la lectura de los CSV locales y (opcional) desde HDFS usando PySpark.

Estructura esperada:
```
.
├── ml-latest-small
│   ├── links.csv
│   ├── movies.csv
│   ├── ratings.csv
│   ├── tags.csv
│   └── README.txt
└── ml-latest-small.zip
```

## 1) Configuración

In [ ]:
import os

# Ajusta estas variables si es necesario
BASE_DIR = os.path.abspath("ml-latest-small")
USE_HDFS = False  # Cambia a True si ya subiste a HDFS (/movielens)

# Si usas instalación manual de Spark, asegúrate de tener SPARK_HOME en el entorno
# y Java configurado. Si usas findspark, descomenta:
try:
    import findspark
    findspark.init()
except Exception as e:
    print("findspark no disponible o no necesario:", e)

os.environ["PYSPARK_PYTHON"] = "python3"

BASE_DIR

## 2) Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder     .appName("MovieLens Test")     .getOrCreate()

spark

## 3) Cargar datos (local o HDFS)

In [ ]:
from pyspark.sql.functions import col

if USE_HDFS:
    prefix = "hdfs:///movielens/"
else:
    prefix = BASE_DIR + "/"

movies = spark.read.csv(prefix + "movies.csv", header=True, inferSchema=True)
ratings = spark.read.csv(prefix + "ratings.csv", header=True, inferSchema=True)
tags = spark.read.csv(prefix + "tags.csv", header=True, inferSchema=True)
links = spark.read.csv(prefix + "links.csv", header=True, inferSchema=True)

(movies.count(), ratings.count(), tags.count(), links.count())

## 4) Esquemas

In [ ]:
movies.printSchema()
ratings.printSchema()
tags.printSchema()
links.printSchema()

## 5) Valores nulos

In [ ]:
from pyspark.sql.functions import sum

def nulls(df):
    return df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])

nulls(movies).show()
nulls(ratings).show()
nulls(tags).show()
nulls(links).show()

## 6) Primeros 10 registros

In [ ]:
movies.show(10, truncate=False)
ratings.show(10, truncate=False)
tags.show(10, truncate=False)
links.show(10, truncate=False)

## 7) Consultas

In [ ]:
from pyspark.sql.functions import split, explode

df = ratings.join(movies, "movieId")

print("Películas con rating 5.0:")
df.filter(col("rating") == 5.0).select("title").distinct().show(truncate=False)

print("Películas con rating <= 0.5:")
df.filter(col("rating") <= 0.5).select("title").distinct().show(truncate=False)

print("Géneros:")
genres = movies.select(explode(split(col("genres"), "\\|")).alias("genre"))
genres.distinct().show()

print("Películas por género:")
genres.groupBy("genre").count().show()